# Phase 4 — Google Vision API OCR Accuracy

**Purpose:** Test Google Vision OCR on 20 real product photos and document failure cases.

**Prerequisites:**
- Docker API running (default http://localhost:8000)
- 20 product JPEGs in `backend/test_photos/`
- Google service account at `backend/credentials/shelf-love-353bbeca17ab.json`

In [14]:
import os, sys, json, time
from pathlib import Path
from io import BytesIO

import requests
import pandas as pd
from IPython.display import display

In [15]:
# === CONFIGURATION ===
API_BASE_URL = os.environ.get("API_BASE_URL", "http://localhost:8000")

# Test user — the notebook will register this user if they don’t exist
TEST_EMAIL = "ocr_test@example.com"
TEST_PASSWORD = "test123"
TEST_NAME = "OCR Test"

# Image directory (this notebook is at backend/notebooks/)
NOTEBOOK_DIR = Path.cwd()
BACKEND_DIR = NOTEBOOK_DIR.parent
IMAGE_DIR = BACKEND_DIR / "test_photos"

IMAGE_EXTS = {".jpg", ".jpeg", ".png"}
image_paths = sorted([p for p in IMAGE_DIR.iterdir() if p.suffix.lower() in IMAGE_EXTS])

print(f"API: {API_BASE_URL}")
print(f"Images dir: {IMAGE_DIR}")
print(f"Images found: {len(image_paths)}")
print(f"Credentials exist: {(BACKEND_DIR / 'credentials' / 'shelf-love-353bbeca17ab.json').exists()}")

API: http://localhost:8000
Images dir: c:\Projects\cosmetic-expiry-scanner\backend\test_photos
Images found: 20
Credentials exist: True


In [16]:
# === LOGIN / REGISTER ===
def login_or_register(email, password, name):
    r = requests.post(f"{API_BASE_URL}/auth/login", json={"email": email, "password": password})
    if r.status_code == 200:
        print(f"Logged in as {email}")
        return r.json()["access_token"]
    print(f"Login failed ({r.status_code}), registering...")
    r = requests.post(f"{API_BASE_URL}/auth/register", json={"email": email, "password": password, "name": name})
    if r.status_code == 201:
        print(f"Registered as {email}, logging in...")
        r = requests.post(f"{API_BASE_URL}/auth/login", json={"email": email, "password": password})
        if r.status_code == 200:
            return r.json()["access_token"]
        raise RuntimeError(f"Login after register failed: {r.status_code} {r.text}")
    raise RuntimeError(f"Auth failed: {r.status_code} {r.text}")

TOKEN = login_or_register(TEST_EMAIL, TEST_PASSWORD, TEST_NAME)
HEADERS = {"Authorization": f"Bearer {TOKEN}"}
print("Token acquired")

Logged in as ocr_test@example.com
Token acquired


In [17]:
# === HELPER: UPLOAD IMAGE + RUN OCR ===
def upload_and_scan(image_path, verbose=True):
    filename = image_path.name
    # 1) Get presigned upload URL
    r = requests.post(f"{API_BASE_URL}/uploads/presigned-url",
        json={"file_name": filename, "content_type": "image/jpeg"}, headers=HEADERS)
    r.raise_for_status()
    ud = r.json()
    if verbose:
        print(f"  presigned URL OK")
    # 2) Upload to S3
    with open(image_path, "rb") as f:
        data = f.read()
    r = requests.put(ud["upload_url"], data=data, headers={"Content-Type": "image/jpeg"})
    if r.status_code not in (200, 201, 204):
        raise RuntimeError(f"S3 upload failed: {r.status_code}")
    if verbose:
        print(f"  uploaded to S3: {ud['file_key']}")
    # 3) Process (OCR)
    r = requests.post(f"{API_BASE_URL}/uploads/process",
        json={"file_key": ud["file_key"], "barcode": None}, headers=HEADERS)
    r.raise_for_status()
    pd = r.json()
    if verbose:
        print(f"  OCR done: {len(pd.get('raw_ocr_text') or '')} chars, scan_id={pd.get('scan_id')}")
    # 4) Get presigned download URL (for possible future use)
    download_url = None
    try:
        r2 = requests.get(f"{API_BASE_URL}/uploads/{ud['file_key']}/url", headers=HEADERS)
        if r2.status_code == 200:
            download_url = r2.json()["download_url"]
    except Exception:
        pass
    return {
        "filename": filename,
        "file_key": ud["file_key"],
        "download_url": download_url,
        "scan_id": pd.get("scan_id"),
        "raw_ocr_text": pd.get("raw_ocr_text") or "",
        "success": True,
    }

In [18]:
# === RUN OCR ON ALL IMAGES ===
results = []
print(f"Processing {len(image_paths)} images...\n")
for i, img_path in enumerate(image_paths, 1):
    print(f"[{i}/{len(image_paths)}] {img_path.name}")
    try:
        res = upload_and_scan(img_path)
        results.append(res)
    except Exception as e:
        print(f"  ERROR: {e}")
        results.append({
            "filename": img_path.name, "file_key": None, "download_url": None,
            "scan_id": None, "raw_ocr_text": "", "success": False,
        })
    print()

ok = sum(1 for r in results if r["success"])
print(f"\nDone. {ok}/{len(results)} succeeded.")

Processing 20 images...

[1/20] 01.jpeg
  presigned URL OK
  uploaded to S3: products/60a1fae7-013c-4259-aea0-bcdfcbb0fc38.jpeg
  OCR done: 299 chars, scan_id=87

[2/20] 02.jpeg
  presigned URL OK
  uploaded to S3: products/995a3ecc-c216-4c85-980f-df9a6d3155a6.jpeg
  OCR done: 2693 chars, scan_id=88

[3/20] 03.jpeg
  presigned URL OK
  uploaded to S3: products/7ce716fd-ed70-4ea0-8b1b-841d104d0376.jpeg
  OCR done: 747 chars, scan_id=89

[4/20] 04.jpeg
  presigned URL OK
  uploaded to S3: products/5809cae5-8d24-4517-9958-fe02945db027.jpeg
  OCR done: 227 chars, scan_id=90

[5/20] 05.jpeg
  presigned URL OK
  uploaded to S3: products/97765831-5260-4ea7-9561-78cca7ec4fe3.jpeg
  OCR done: 1682 chars, scan_id=91

[6/20] 06.jpeg
  presigned URL OK
  uploaded to S3: products/f4a2cf3c-defd-4c9a-80de-5072d6c381e3.jpeg
  OCR done: 144 chars, scan_id=92

[7/20] 07.jpeg
  presigned URL OK
  uploaded to S3: products/2d5b3d3a-aff7-4603-b832-fdb167ae5f55.jpeg
  OCR done: 1641 chars, scan_id=93

[8/20]

In [19]:
# === SUMMARY TABLE ===
df = pd.DataFrame(results)
df["chars"] = df["raw_ocr_text"].apply(len)
df["preview"] = df["raw_ocr_text"].apply(
    lambda t: t.replace("\n", " ") if t else ""
)
display(df[["filename", "success", "scan_id", "chars", "preview"]])

,filename,success,scan_id,chars,preview
0,01.jpeg,True,87,299,BIODERMA LABORATOIRE DERMATOLOGIQUE SENSIBIO H...
1,02.jpeg,True,88,2693,Physiob Unfragance DAF Advancedon esmicellaire...
2,03.jpeg,True,89,747,"INGREDIENTS: AQUA, ALUMINUM CHLOROHYDRATE, PPG..."
3,04.jpeg,True,90,227,and 2568 88 sesderma DRYSES Deodorant roll-on ...
4,05.jpeg,True,91,1682,Long Damaged Hair? based amino acids similar t...
5,06.jpeg,True,92,144,L L'ORÉAL PARIS ELVIVE DREAM LENGTHS SPLIT END...
6,07.jpeg,True,93,1641,APPLE CIDER VINEGAR BLEND CLARIFY & SHINE SHAM...
7,08.jpeg,True,94,345,NEW LOOKI SAME GREAT FORMULA Aveeno SCALP SOOT...
8,09.jpeg,True,95,433,X2M HYGLAM CONCEALER Brightening & Hydrating C...
9,10.jpeg,True,96,17,NATASHA DENONA ND


In [20]:
# === OCR TEXT DUMP ===
for res in results:
    print("=" * 60)
    print(f"FILE: {res['filename']}  (scan_id={res['scan_id']})")
    print("-" * 60)
    print(res["raw_ocr_text"] if res["raw_ocr_text"] else "(no text detected)")
    print()

FILE: 01.jpeg  (scan_id=87)
------------------------------------------------------------
BIODERMA
LABORATOIRE DERMATOLOGIQUE
SENSIBIO
H2O
PEAUX SENSIBLES
SENSITIVE SKIN
L'eau micellaire Originale
Nettoie Démaquille - Apaise
Visage - Yeux
The Original micellar water
Cleanses - Removes makeup - Soothes
Face - Eyes
NAOS Micellar
ECOBIOLOGY technology
500 ml e 16.9 FL. OZ.
Made in France
m

FILE: 02.jpeg  (scan_id=88)
------------------------------------------------------------
Physiob
Unfragance
DAF
Advancedon
esmicellaire Sensibio H2O garantit un nettoyage et un démaquillage en douceur du pH o
et des yeux et favorise l'élimination des résidus de pollution et de pollen. Non p
La technologie micellaire, inspirée par notre démarche unique NAOS ÉCOBIOLOGIE
elimine les impuretes tout en respectant l'équilibre cutané. Les actifs dermatologiques
apaisants aident à prevenir les sensations d'irritation. Testé sous contrôle dermatologique
et ophtalmologique. EFFICACITÉ PROUVÉE: nettoie en douceur,

In [21]:
# === CHECK THE DATABASE ===
ids = [str(r["scan_id"]) for r in results if r["scan_id"] is not None]
if ids:
    id_list = ", ".join(ids)
    print("Run this to inspect stored records:\n")
    print(f"docker compose -f docker-compose.dev.yml exec db psql "
          f"-U cosmetics_user -d cosmetics "
          f"-c \"SELECT id, image_s3_key, LEFT(raw_ocr_text, 200) "
          f"FROM scan_results WHERE id IN ({id_list}) ORDER BY id;\"")
else:
    print("No scan IDs to inspect.")

Run this to inspect stored records:

docker compose -f docker-compose.dev.yml exec db psql -U cosmetics_user -d cosmetics -c "SELECT id, image_s3_key, LEFT(raw_ocr_text, 200) FROM scan_results WHERE id IN (87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106) ORDER BY id;"


---
## Failure Case Documentation

After reviewing the OCR text dump above, fill in the table below for every image where
OCR produced poor or missing results.

| # | Filename | Failure Type | Observed Issue | Would Phase 5 fail? |
|---|----------|-------------|----------------|---------------------|
| 1 | 02.jpeg | low_contrast | Transparent label on transparent bottle; some text not picked up; PAO symbol out of frame and not detected | Yes — PAO symbol absent means no period-after-opening date |
| 2 | 04.jpeg | curved_surface | Round label; edge text clipped due to label curvature | Partial — center text may parse, but edge fields (often expiry/batch) would be missed |
| 3 | 09.jpeg | small_font, blurry | Other text detected successfully; PAO date too small and blurry to read | Yes — PAO date not extracted |

**Failure type options:** `reflection`, `curved_surface`, `small_font`,
`low_contrast`, `angled`, `multi_language`, `blurry`, `other`

### Summary

**Overall accuracy:** (how many images produced usable text?) 17/20 (3 partial or failed to detect PAO)

**Most common failure:** `small_font` / `blurry` — small PAO dates are the recurring blind spot across failure cases

**Key insight for Phase 5:** PAO symbol detection cannot rely on OCR text alone — the symbol may be out of frame (02.jpeg) or the adjacent date too small to read cleanly (09.jpeg).

---
*Save this notebook after filling in failure documentation.*